In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from sklearn.metrics import precision_recall_fscore_support
import numpy as np
import os
import sys

In [2]:
# Đặt seed để đảm bảo kết quả có thể lặp lại
torch.manual_seed(42)

# Thiết lập Device (Kaggle thường có GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Sử dụng thiết bị: {device}")

Sử dụng thiết bị: cuda


In [5]:
# --- THIẾT LẬP DỮ LIỆU KAGGLE ---
KAGGLE_INPUT_DIR = '/kaggle/input/vinafood21-zip/VinaFood21' 
DATA_DIR = os.path.join(KAGGLE_INPUT_DIR)

NUM_CLASSES = 21 

# --- KIỂM TRA CẤU TRÚC CUỐI CÙNG ---
if os.path.exists(os.path.join(DATA_DIR, 'train')):
    print(f"\n[OK]: Cấu trúc thư mục đã được xác nhận. DATA_DIR: {DATA_DIR}")
else:
    print(f"\n[LỖI]: Không tìm thấy thư mục 'train' trong DATA_DIR đã định: {DATA_DIR}")
    print("Vui lòng kiểm tra tên thư mục (VinaFood21) có đúng không.")
    sys.exit()

print(f"DATA_DIR cuối cùng được sử dụng để tải dữ liệu: {DATA_DIR}")


[OK]: Cấu trúc thư mục đã được xác nhận. DATA_DIR: /kaggle/input/vinafood21-zip/VinaFood21
DATA_DIR cuối cùng được sử dụng để tải dữ liệu: /kaggle/input/vinafood21-zip/VinaFood21


In [6]:
# Đặt seed để đảm bảo kết quả có thể lặp lại
torch.manual_seed(42)

# --- Lớp Inception Module ---
class Inception(nn.Module):
    def __init__(self, in_channels, n1x1, n3x3red, n3x3, n5x5red, n5x5, pool_proj):
        super(Inception, self).__init__()

        # Branch 1: 1x1 Conv
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, n1x1, kernel_size=1), nn.ReLU(True)
        )
        # Branch 2: 1x1 Conv -> 3x3 Conv
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, n3x3red, kernel_size=1), nn.ReLU(True),
            nn.Conv2d(n3x3red, n3x3, kernel_size=3, padding=1), nn.ReLU(True)
        )
        # Branch 3: 1x1 Conv -> 5x5 Conv
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, n5x5red, kernel_size=1), nn.ReLU(True),
            nn.Conv2d(n5x5red, n5x5, kernel_size=5, padding=2), nn.ReLU(True)
        )
        # Branch 4: 3x3 Max Pool -> 1x1 Conv
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1, ceil_mode=True),
            nn.Conv2d(in_channels, pool_proj, kernel_size=1), nn.ReLU(True)
        )

    def forward(self, x):
        return torch.cat([self.branch1(x), self.branch2(x), self.branch3(x), self.branch4(x)], 1)

# --- Lớp GoogLeNet Model ---
class GoogLeNet(nn.Module):
    def __init__(self, num_classes=1000):
        super(GoogLeNet, self).__init__()

        # C1: Conv 7x7/2, padding=3 (theo yêu cầu)
        self.conv1 = nn.Sequential(nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3), nn.ReLU(True))
        # S1: Max Pool 3x3/2, ceil_mode=True (theo yêu cầu)
        self.maxpool1 = nn.MaxPool2d(kernel_size=3, stride=2, padding=0, ceil_mode=True)

        # C2, S2, Inception 3(a, b), S3
        self.conv2 = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=1), nn.ReLU(True),
            nn.Conv2d(64, 192, kernel_size=3, padding=1), nn.ReLU(True)
        )
        self.maxpool2 = nn.MaxPool2d(kernel_size=3, stride=2, padding=0, ceil_mode=True)
        self.inception3a = Inception(192, 64, 96, 128, 16, 32, 32)
        self.inception3b = Inception(256, 128, 128, 192, 32, 96, 64)
        self.maxpool3 = nn.MaxPool2d(kernel_size=3, stride=2, padding=0, ceil_mode=True)

        # Inception 4(a, b, c, d, e), S4
        self.inception4a = Inception(480, 192, 96, 208, 16, 48, 64)
        self.inception4b = Inception(512, 160, 112, 224, 24, 64, 64)
        self.inception4c = Inception(512, 128, 128, 256, 24, 64, 64)
        self.inception4d = Inception(512, 112, 144, 288, 32, 64, 64)
        self.inception4e = Inception(528, 256, 160, 320, 32, 128, 128)
        self.maxpool4 = nn.MaxPool2d(kernel_size=3, stride=2, padding=0, ceil_mode=True)

        # Inception 5(a, b)
        self.inception5a = Inception(832, 256, 160, 320, 32, 128, 128)
        self.inception5b = Inception(832, 384, 192, 384, 48, 128, 128)

        # Output Layers
        self.avgpool = nn.AvgPool2d(kernel_size=7, stride=1)
        self.dropout = nn.Dropout(p=0.4)
        self.linear = nn.Linear(1024, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.maxpool1(x)
        x = self.conv2(x)
        x = self.maxpool2(x)
        x = self.inception3a(x)
        x = self.inception3b(x)
        x = self.maxpool3(x)
        x = self.inception4a(x)
        x = self.inception4b(x)
        x = self.inception4c(x)
        x = self.inception4d(x)
        x = self.inception4e(x)
        x = self.maxpool4(x)
        x = self.inception5a(x)
        x = self.inception5b(x)

        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = self.linear(x)
        return x

In [7]:
# --- Tải Dữ liệu VinaFood21 (ImageFolder) ---

# Dữ liệu ảnh màu (3 kênh) cần được resize về 224x224 cho GoogLeNet
transform_train = transforms.Compose([
    # Bước 1: Resize cạnh ngắn nhất về 256 (hoặc một giá trị lớn hơn 224)
    transforms.Resize(256),
    # Bước 2: Cắt ngẫu nhiên để lấy chính xác 224x224
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

transform_test = transforms.Compose([
    # Bước 1: Resize cạnh ngắn nhất về 256
    transforms.Resize(256),
    # Bước 2: Cắt chính giữa để lấy chính xác 224x224
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print(f"Đang tải dữ liệu từ thư mục: {DATA_DIR}")
try:
    train_dataset = torchvision.datasets.ImageFolder(
        root=os.path.join(DATA_DIR, 'train'), transform=transform_train
    )
    test_dataset = torchvision.datasets.ImageFolder(
        root=os.path.join(DATA_DIR, 'test'), transform=transform_test
    )
except Exception as e:
    print(f"\n[LỖI]: Không thể tải dữ liệu. Kiểm tra DATA_DIR và cấu trúc thư mục. Lỗi: {e}")
    exit()

Đang tải dữ liệu từ thư mục: /kaggle/input/vinafood21-zip/VinaFood21


In [8]:
# Kiểm tra số lớp thực tế và cập nhật mô hình
NUM_CLASSES = len(train_dataset.classes)
print(f"Số lớp thực tế tìm thấy: {NUM_CLASSES}")

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

Số lớp thực tế tìm thấy: 21


In [9]:
# Khởi tạo mô hình
model = GoogLeNet(num_classes=NUM_CLASSES).to(device)

In [10]:
# --- Thiết lập Huấn luyện ---
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
NUM_EPOCHS = 10

In [11]:
# --- Huấn luyện Mô hình ---
print(f"\nBắt đầu huấn luyện mô hình GoogLeNet trên {device}...")

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        if (i+1) % 50 == 0:
            print(f"Epoch {epoch+1}/{NUM_EPOCHS}, Step {i+1}/{len(train_loader)}, Loss: {loss.item():.4f}")

    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}, Average Loss: {epoch_loss:.4f}")

print("Huấn luyện hoàn tất.")


Bắt đầu huấn luyện mô hình GoogLeNet trên cuda...
Epoch 1/10, Step 50/314, Loss: 2.9057
Epoch 1/10, Step 100/314, Loss: 3.0376
Epoch 1/10, Step 150/314, Loss: 3.0052
Epoch 1/10, Step 200/314, Loss: 3.0014
Epoch 1/10, Step 250/314, Loss: 3.0028


/usr/local/lib/python3.11/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 1/10, Step 300/314, Loss: 2.9799
Epoch 1/10, Average Loss: 2.9862
Epoch 2/10, Step 50/314, Loss: 3.0020
Epoch 2/10, Step 100/314, Loss: 2.9244
Epoch 2/10, Step 150/314, Loss: 2.9848
Epoch 2/10, Step 200/314, Loss: 3.0420
Epoch 2/10, Step 250/314, Loss: 3.0235
Epoch 2/10, Step 300/314, Loss: 2.9647
Epoch 2/10, Average Loss: 2.9800
Epoch 3/10, Step 50/314, Loss: 2.9942
Epoch 3/10, Step 100/314, Loss: 2.8778
Epoch 3/10, Step 150/314, Loss: 2.9872
Epoch 3/10, Step 200/314, Loss: 2.9344
Epoch 3/10, Step 250/314, Loss: 2.8767
Epoch 3/10, Step 300/314, Loss: 2.9578
Epoch 3/10, Average Loss: 2.9778
Epoch 4/10, Step 50/314, Loss: 2.9574
Epoch 4/10, Step 100/314, Loss: 3.0103
Epoch 4/10, Step 150/314, Loss: 2.8689
Epoch 4/10, Step 200/314, Loss: 3.0118
Epoch 4/10, Step 250/314, Loss: 2.9650
Epoch 4/10, Step 300/314, Loss: 2.9387
Epoch 4/10, Average Loss: 2.9858
Epoch 5/10, Step 50/314, Loss: 3.0696
Epoch 5/10, Step 100/314, Loss: 2.9134
Epoch 5/10, Step 150/314, Loss: 3.0113
Epoch 5/10, St

In [12]:
# --- Đánh giá Mô hình trên các Độ đo Yêu cầu ---
model.eval()
all_preds = []
all_labels = []
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Tính Accuracy
accuracy = 100 * correct / total
print(f"\nAccuracy tổng thể của mô hình trên tập kiểm tra: {accuracy:.2f}%")

# Tính các độ đo Precision, Recall, F1-macro (Yêu cầu)
precision, recall, f1, _ = precision_recall_fscore_support(
    all_labels,
    all_preds,
    average='macro',
    labels=np.arange(NUM_CLASSES),
    zero_division=0
)

print("\n--- Kết quả Đánh giá Độ đo Macro (Precision, Recall, F1) ---")
print(f"Precision (Macro): {precision:.4f}")
print(f"Recall (Macro):    {recall:.4f}")
print(f"F1-score (Macro):  {f1:.4f}")


Accuracy tổng thể của mô hình trên tập kiểm tra: 9.08%

--- Kết quả Đánh giá Độ đo Macro (Precision, Recall, F1) ---
Precision (Macro): 0.0043
Recall (Macro):    0.0476
F1-score (Macro):  0.0079
